# Isolation Forest baseline

This notebook reviews the validated feature dataset, fits a baseline Isolation Forest on the chronological training period only, scores all partitions, and evaluates known red-team overlap after training. Red-team data is never used as a training feature or filter.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_redteam_events
from src.detector import FEATURE_COLUMNS, MODEL_FEATURE_COLUMNS, IsolationForestDetector
from src.features import chronological_split

FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_train.parquet'
REDTEAM_PATH = PROJECT_ROOT / 'data' / 'raw' / 'redteam.txt.gz'
ARTIFACT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'isolation_forest_baseline.joblib'
CONTAMINATION = 'auto'
RANDOM_STATE = 42

features = pd.read_parquet(FEATURE_PATH)
redteam = load_redteam_events(REDTEAM_PATH)
train, validation, test = chronological_split(features)
print('Loaded:', features.shape)
print('Chronological split:', len(train), len(validation), len(test))


## Final feature review

All original behavioral features are reviewed. The baseline retains one representative from each highly redundant group; no source column is modified.

In [ ]:
review = features[FEATURE_COLUMNS].describe().T
review['dtype'] = features[FEATURE_COLUMNS].dtypes.astype(str)
review['unique_values'] = features[FEATURE_COLUMNS].nunique()
display(review[['dtype', 'unique_values', 'min', 'max', 'mean', 'std']])

correlation = features[FEATURE_COLUMNS].corr()
redundant_pairs = []
for index, left in enumerate(FEATURE_COLUMNS):
    for right in FEATURE_COLUMNS[index + 1:]:
        value = float(correlation.loc[left, right])
        if abs(value) >= 0.99:
            redundant_pairs.append((left, right, value))
print('Highly correlated pairs (absolute Pearson correlation >= 0.99):')
for left, right, value in redundant_pairs:
    print(f'  {left} <-> {right}: {value:.6f}')

print('\nRecommended retained baseline features:', MODEL_FEATURE_COLUMNS)
print('Excluded from baseline as redundant:', sorted(set(FEATURE_COLUMNS) - set(MODEL_FEATURE_COLUMNS)))


## Fit and score

Only `MODEL_FEATURE_COLUMNS` enter the matrix. Entity, timestamps, and red-team fields remain evaluation/context data. The anomaly score is `-decision_function`, so larger values are more suspicious.

In [ ]:
detector = IsolationForestDetector(
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_estimators=200,
)
detector.fit(train)
detector.save(ARTIFACT_PATH)

scored = {}
for name, frame in {'train': train, 'validation': validation, 'test': test}.items():
    scored[name] = frame.join(detector.detect(frame))
    scored[name]['is_anomaly'] = scored[name]['is_anomaly'].astype(bool)
    print(f'{name} anomaly rate: {scored[name].is_anomaly.mean():.4%}')

print('Features used:', MODEL_FEATURE_COLUMNS)
print('Features excluded:', sorted(set(FEATURE_COLUMNS) - set(MODEL_FEATURE_COLUMNS)))
print('Saved artifact:', ARTIFACT_PATH)


In [ ]:
score_associations = train[MODEL_FEATURE_COLUMNS].corrwith(scored['train']['anomaly_score']).abs().sort_values(ascending=False)
print('Absolute Pearson association with training anomaly score (not feature importance):')
display(score_associations.rename('absolute_correlation').to_frame())
print('Anomaly threshold:', detector.anomaly_threshold)
print('Risk thresholds (MEDIUM, HIGH, CRITICAL):', detector.risk_thresholds)


## Post-training red-team evaluation

A test row is red-team-related only when its entity participates in a known red-team event whose timestamp falls inside that row's feature window. Non-related rows are an evaluation approximation, not guaranteed benign labels.

In [ ]:
def redteam_related(frame, events):
    related = pd.Series(False, index=frame.index)
    covered_events = 0
    for event in events.itertuples(index=False):
        entity_match = frame['entity'].isin([str(event.user), str(event.source_computer), str(event.destination_computer)])
        time_match = frame['window_start'].le(int(event.timestamp)) & frame['window_end'].ge(int(event.timestamp))
        event_rows = entity_match & time_match
        if event_rows.any():
            covered_events += 1
            related |= event_rows
    return related, covered_events

related_test, covered_events = redteam_related(test, redteam)
test_scored = scored['test'].copy()
test_scored['redteam_related'] = related_test.to_numpy()
y_true = test_scored['redteam_related'].to_numpy(dtype=int)
y_pred = test_scored['is_anomaly'].to_numpy(dtype=int)

redteam_windows = int(related_test.sum())
detected_redteam_windows = int((related_test & test_scored['is_anomaly']).sum())
non_redteam = ~related_test
metrics = {
    'precision': precision_score(y_true, y_pred, zero_division=0),
    'recall': recall_score(y_true, y_pred, zero_division=0),
    'f1': f1_score(y_true, y_pred, zero_division=0),
}
print('Red-team events covered by test windows:', covered_events)
print('Red-team-related test windows:', redteam_windows)
print('Detected red-team windows:', detected_redteam_windows)
print('Detection rate:', detected_redteam_windows / redteam_windows if redteam_windows else None)
print('Anomaly rate among red-team-related windows:', test_scored.loc[related_test, 'is_anomaly'].mean() if related_test.any() else None)
print('Anomaly rate among non-red-team windows:', test_scored.loc[non_redteam, 'is_anomaly'].mean() if non_redteam.any() else None)
print('Precision:', metrics['precision'])
print('Recall:', metrics['recall'])
print('F1:', metrics['f1'])
print('Confusion matrix [normal, red-team-related]:')
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


## Most anomalous windows and final report

In [ ]:
print(test_scored.sort_values('anomaly_score', ascending=False)[['timestamp', 'entity', 'anomaly_score', 'risk_level', 'is_anomaly']].head(20).to_string(index=False))
print('''
========================================
ISOLATION FOREST BASELINE
========================================
Training rows:''', len(train))
print('Validation rows:', len(validation))
print('Test rows:', len(test))
print('Features used:', MODEL_FEATURE_COLUMNS)
print('Features excluded:', sorted(set(FEATURE_COLUMNS) - set(MODEL_FEATURE_COLUMNS)))
print('Contamination:', CONTAMINATION)
print('Random state:', RANDOM_STATE)
print('Train anomaly rate:', f"{scored['train'].is_anomaly.mean():.4%}")
print('Validation anomaly rate:', f"{scored['validation'].is_anomaly.mean():.4%}")
print('Test anomaly rate:', f"{scored['test'].is_anomaly.mean():.4%}")
print('Red-team windows:', redteam_windows)
print('Detected red-team windows:', detected_redteam_windows)
print('Detection rate:', f"{detected_redteam_windows / redteam_windows:.4%}" if redteam_windows else 'N/A')
print('Precision:', f"{metrics['precision']:.4f}")
print('Recall:', f"{metrics['recall']:.4f}")
print('F1:', f"{metrics['f1']:.4f}")
print('========================================')
